# Celeb-DF-v2 전체 ArcFace 얼굴인식 평가 — 딥소각 얼굴가드

이 노트북은 **Celeb-real 590개 영상 전체**를 처리한다. 각 영상에서 10개 프레임을 균등 추출하고, 얼굴 탐지·정렬·ArcFace 추론 후 프레임 임베딩을 평균하여 **영상당 하나의 임베딩**을 만든다. 같은 영상의 프레임이 등록과 테스트에 동시에 들어가지 않는다.

- 원본 규모: 590개 영상, 59명, 약 946.5MB
- 기본 처리량: 최대 5,900개 프레임
- 최종 평가: 등록 영상 5개 + 테스트 영상 3개 이상이 가능한 56명
- 프로토콜: 등록 3개 영상 / 등록 5개 영상, query는 두 프로토콜 모두 6번째 영상부터 사용
- 검증/테스트: 인물 ID가 겹치지 않는 30% / 70% subject-disjoint 분할
- 지표: ROC-AUC, EER, validation에서 고정한 threshold의 TAR/FAR/FRR, 95% subject bootstrap CI

이 실험은 **일반 얼굴 동일인 검증**이며 딥페이크 탐지 정확도가 아니다. AI-Hub 데이터는 승인 전 사용하지 않는다.

InsightFace 코드는 MIT이지만 제공 사전학습 모델은 비상업 연구 용도이다. 해커톤 연구 검증에만 사용하고 제품에 그대로 탑재하지 않는다.

In [ ]:
#@title 1. 실행 설정과 권한 확인
REPO_URL = "https://github.com/Chunbae-A/face-image.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
CODE_SOURCE = "embedded" #@param ["embedded", "github"]
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
SOURCE_ZIP_PATH = "/content/drive/MyDrive/Celeb-DF-v2.zip" #@param {type:"string"}
COPY_ZIP_TO_RUNTIME = False #@param {type:"boolean"}
PERSIST_DERIVED_RESULTS_TO_DRIVE = False #@param {type:"boolean"}
DRIVE_RESULT_DIR = "/content/drive/MyDrive/face-image-celebdf-results" #@param {type:"string"}

# 공식 신청·승인으로 받은 파일이며, 해당 약관상 Colab/Drive 처리가 허용되는지 직접 확인 후 True로 변경합니다.
I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED = False #@param {type:"boolean"}
# InsightFace 제공 buffalo_l 가중치는 비상업 연구 전용입니다.
I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE = False #@param {type:"boolean"}

RUN_SMOKE_BEFORE_FULL = True #@param {type:"boolean"}
RUN_FULL_590_VIDEOS = True #@param {type:"boolean"}
FRAMES_PER_VIDEO = 10 #@param {type:"integer"}
MINIMUM_VALID_FRAMES = 3 #@param {type:"integer"}
BOOTSTRAP_REPEATS = 500 #@param {type:"integer"}
SEED = 20260805 #@param {type:"integer"}

import sys
IN_HOSTED_COLAB = "google.colab" in sys.modules
if IN_HOSTED_COLAB and not I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED:
    raise PermissionError(
        "Hosted Colab/Drive processing is blocked until the Celeb-DF terms are checked. "
        "After checking them, set I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED=True."
    )
if not I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE:
    raise PermissionError(
        "Review the InsightFace model license, then set "
        "I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE=True."
    )
if not RUN_FULL_590_VIDEOS:
    raise ValueError("This notebook is configured for the requested full 590-video run.")
print({
    "hosted_colab": IN_HOSTED_COLAB,
    "run_full": RUN_FULL_590_VIDEOS,
    "frames_per_video": FRAMES_PER_VIDEO,
    "maximum_frame_inferences": 590 * FRAMES_PER_VIDEO,
})

## 권장 실행 환경

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한다. 아래 설치는 2026-08-05 기준 공식 PyPI의 `insightface==1.0.1`을 사용한다. 설치 후 ONNX Runtime import 오류가 나면 런타임을 한 번 다시 시작하고 1번 셀부터 재실행한다.

In [ ]:
#@title 2. 라이브러리 설치
%pip uninstall -y -q onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir "insightface==1.0.1" "onnxruntime-gpu==1.23.2" opencv-python-headless pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
#@title 3. 실행 코드 준비 — 기본값은 GitHub 권한이 필요 없는 내장 모드
from pathlib import Path
import base64
import os
import subprocess

EMBEDDED_FILES_B64 = {'scripts/celebdf_faceguard.py': '...', 'scripts/run_celebdf_arcface.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFeHRyYWN0IHZpZGVvLWxldmVsIEFyY0ZhY2UgZW1iZWRkaW5ncyBmcm9tIENlbGViLURGLXYyIENlbGViLXJlYWwgdmlkZW9zLgoKVGhpcyBydW5uZXIgaXMgaW50ZW5kZWQgZm9yIEdvb2dsZSBDb2xhYiBvciBhbm90aGVyIGVudmlyb25tZW50IHdpdGggT3BlbkNWLApJbnNpZ2h0RmFjZSwgYW5kIE9OTlggUnVudGltZSBpbnN0YWxsZWQuICBJdCBuZXZlciBzYXZlcyBmYWNlIGNyb3BzIG9yIHNhbXBsZWQKZnJhbWVzLiAgRXZlcnkgc3VjY2Vzc2Z1bCB2aWRlbyBwcm9kdWNlcyBvbmUgbm9ybWFsaXplZCA1MTItRCBlbWJlZGRpbmcsIGFuZAp0aGUgTlBaIGNoZWNrcG9pbnQgaXMgYXRvbWljYWxseSByZXBsYWNlZCBhdCBhIGNvbmZpZ3VyYWJsZSBpbnRlcnZhbC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0CmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBjZWxlYmRmX2ZhY2VndWFyZCBpbXBvcnQgKAogICAgQXJjaGl2ZVZpZGVvLAogICAgVmlkZW9FbWJlZGRpbmcsCiAgICBsMl9ub3JtYWxpemUsCiAgICBsb2FkX3ZpZGVvX2VtYmVkZGluZ3MsCiAgICByZWFkX21hbmlmZXN0LAogICAgc2F2ZV92aWRlb19lbWJlZGRpbmdzLAogICAgc2VsZWN0X3Ntb2tlX3Jvd3MsCikKCgpkZWYgc2FtcGxlX2ZyYW1lX2luZGljZXMoZnJhbWVfY291bnQ6IGludCwgcmVxdWVzdGVkOiBpbnQpIC0+IGxpc3RbaW50XToKICAgICIiIlJldHVybiB1bmlxdWUsIGV2ZW5seSBzcGFjZWQgZnJhbWUgaW5kaWNlcyB3aGlsZSBhdm9pZGluZyBoYXJkIGN1dHMgYXQgZW5kcy4iIiIKICAgIGlmIGZyYW1lX2NvdW50IDw9IDAgb3IgcmVxdWVzdGVkIDw9IDA6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBmcmFtZV9jb3VudCA8PSByZXF1ZXN0ZWQ6CiAgICAgICAgcmV0dXJuIGxpc3QocmFuZ2UoZnJhbWVfY291bnQpKQogICAgZmlyc3QgPSBtaW4oZnJhbWVfY291bnQgLSAxLCBtYXgoMCwgaW50KHJvdW5kKGZyYW1lX2NvdW50ICogMC4wOCkpKSkKICAgIGxhc3QgPSBtYXgoZmlyc3QsIG1pbihmcmFtZV9jb3VudCAtIDEsIGludChyb3VuZChmcmFtZV9jb3VudCAqIDAuOTIpKSAtIDEpKQogICAgaW5kaWNlcyA9IG5wLmxpbnNwYWNlKGZpcnN0LCBsYXN0LCBudW09cmVxdWVzdGVkLCBkdHlwZT1pbnQpCiAgICByZXR1cm4gc29ydGVkKHNldChpbnQoaW5kZXgpIGZvciBpbmRleCBpbiBpbmRpY2VzKSkKCgpkZWYgX2ZhY2VfYXJlYV9yYXRpbyhmYWNlOiBBbnksIGZyYW1lX3NoYXBlOiBTZXF1ZW5jZVtpbnRdKSAtPiBmbG9hdDoKICAgIGhlaWdodCwgd2lkdGggPSBpbnQoZnJhbWVfc2hhcGVbMF0pLCBpbnQoZnJhbWVfc2hhcGVbMV0pCiAgICBpZiBoZWlnaHQgPD0gMCBvciB3aWR0aCA8PSAwOgogICAgICAgIHJldHVybiAwLjAKICAgIGxlZnQsIHRvcCwgcmlnaHQsIGJvdHRvbSA9IFtmbG9hdCh2YWx1ZSkgZm9yIHZhbHVlIGluIGZhY2UuYmJveF0KICAgIGFyZWEgPSBtYXgoMC4wLCByaWdodCAtIGxlZnQpICogbWF4KDAuMCwgYm90dG9tIC0gdG9wKQogICAgcmV0dXJuIGFyZWEgLyBmbG9hdChoZWlnaHQgKiB3aWR0aCkKCgpkZWYgc2VsZWN0X3ByaW1hcnlfZmFjZSgKICAgIGZhY2VzOiBTZXF1ZW5jZVtBbnldLAogICAgZnJhbWVfc2hhcGU6IFNlcXVlbmNlW2ludF0sCiAgICBydW5uaW5nX3RlbXBsYXRlOiBucC5uZGFycmF5IHwgTm9uZSwKKSAtPiBBbnkgfCBOb25lOgogICAgIiIiQ2hvb3NlIHRoZSBsYXJnZXN0IGZpcnN0IGZhY2UsIHRoZW4gdHJhY2sgYnkgZW1iZWRkaW5nIHNpbWlsYXJpdHkuIiIiCiAgICBjYW5kaWRhdGVzID0gWwogICAgICAgIGZhY2UgZm9yIGZhY2UgaW4gZmFjZXMgaWYgZ2V0YXR0cihmYWNlLCAibm9ybWVkX2VtYmVkZGluZyIsIE5vbmUpIGlzIG5vdCBOb25lCiAgICBdCiAgICBpZiBub3QgY2FuZGlkYXRlczoKICAgICAgICByZXR1cm4gTm9uZQogICAgaWYgcnVubmluZ190ZW1wbGF0ZSBpcyBOb25lOgogICAgICAgIHJldHVybiBtYXgoY2FuZGlkYXRlcywga2V5PWxhbWJkYSBmYWNlOiBfZmFjZV9hcmVhX3JhdGlvKGZhY2UsIGZyYW1lX3NoYXBlKSkKICAgIHRlbXBsYXRlID0gbDJfbm9ybWFsaXplKHJ1bm5pbmdfdGVtcGxhdGUpCiAgICByZXR1cm4gbWF4KAogICAgICAgIGNhbmRpZGF0ZXMsCiAgICAgICAga2V5PWxhbWJkYSBmYWNlOiBmbG9hdChsMl9ub3JtYWxpemUoZmFjZS5ub3JtZWRfZW1iZWRkaW5nKSBAIHRlbXBsYXRlKSwKICAgICkKCgpkZWYgZW1iZWRfdmlkZW8oCiAgICB2aWRlb19wYXRoOiBQYXRoLAogICAgcm93OiBBcmNoaXZlVmlkZW8sCiAgICBmYWNlX2FwcDogQW55LAogICAgKiwKICAgIGZyYW1lc19wZXJfdmlkZW86IGludCwKICAgIG1pbmltdW1fdmFsaWRfZnJhbWVzOiBpbnQsCikgLT4gdHVwbGVbVmlkZW9FbWJlZGRpbmcgfCBOb25lLCBkaWN0W3N0ciwgb2JqZWN0XSB8IE5vbmVdOgogICAgaW1wb3J0IGN2MiAgIyB0eXBlOiBpZ25vcmUKCiAgICBjYXB0dXJlID0gY3YyLlZpZGVvQ2FwdHVyZShzdHIodmlkZW9fcGF0aCkpCiAgICBpZiBub3QgY2FwdHVyZS5pc09wZW5lZCgpOgogICAgICAgIHJldHVybiBOb25lLCB7InZpZGVvX2lkIjogcm93LnZpZGVvX2lkLCAicmVhc29uIjogInZpZGVvX29wZW5fZmFpbGVkIn0KICAgIHRyeToKICAgICAgICBmcmFtZV9jb3VudCA9IGludChjYXB0dXJlLmdldChjdjIuQ0FQX1BST1BfRlJBTUVfQ09VTlQpKQogICAgICAgIGluZGljZXMgPSBzYW1wbGVfZnJhbWVfaW5kaWNlcyhmcmFtZV9jb3VudCwgZnJhbWVzX3Blcl92aWRlbykKICAgICAgICBpZiBub3QgaW5kaWNlczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIHsidmlkZW9faWQiOiByb3cudmlkZW9faWQsICJyZWFzb24iOiAiaW52YWxpZF9mcmFtZV9jb3VudCJ9CgogICAgICAgIGVtYmVkZGluZ3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgICAgIGRldGVjdGlvbl9zY29yZXM6IGxpc3RbZmxvYXRdID0gW10KICAgICAgICBmYWNlX2FyZWFfcmF0aW9zOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgZGVjb2RlX3NlY29uZHMgPSAwLjAKICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcyA9IDAuMAogICAgICAgIGZvciBmcmFtZV9pbmRleCBpbiBpbmRpY2VzOgogICAgICAgICAgICBkZWNvZGVfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGNhcHR1cmUuc2V0KGN2Mi5DQVBfUFJPUF9QT1NfRlJBTUVTLCBmcmFtZV9pbmRleCkKICAgICAgICAgICAgb2ssIGZyYW1lID0gY2FwdHVyZS5yZWFkKCkKICAgICAgICAgICAgZGVjb2RlX3NlY29uZHMgKz0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIGRlY29kZV9zdGFydAogICAgICAgICAgICBpZiBub3Qgb2sgb3IgZnJhbWUgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBpbmZlcmVuY2Vfc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZhY2VzID0gZmFjZV9hcHAuZ2V0KGZyYW1lKQogICAgICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcyArPSB0aW1lLnBlcmZfY291bnRlcigpIC0gaW5mZXJlbmNlX3N0YXJ0CiAgICAgICAgICAgIHJ1bm5pbmdfdGVtcGxhdGUgPSAoCiAgICAgICAgICAgICAgICBsMl9ub3JtYWxpemUobnAubWVhbihucC5zdGFjayhlbWJlZGRpbmdzKSwgYXhpcz0wKSkKICAgICAgICAgICAgICAgIGlmIGVtYmVkZGluZ3MKICAgICAgICAgICAgICAgIGVsc2UgTm9uZQogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGVjdGVkID0gc2VsZWN0X3ByaW1hcnlfZmFjZShmYWNlcywgZnJhbWUuc2hhcGUsIHJ1bm5pbmdfdGVtcGxhdGUpCiAgICAgICAgICAgIGlmIHNlbGVjdGVkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBlbWJlZGRpbmdzLmFwcGVuZChsMl9ub3JtYWxpemUoc2VsZWN0ZWQubm9ybWVkX2VtYmVkZGluZykpCiAgICAgICAgICAgIGRldGVjdGlvbl9zY29yZXMuYXBwZW5kKGZsb2F0KGdldGF0dHIoc2VsZWN0ZWQsICJkZXRfc2NvcmUiLCBucC5uYW4pKSkKICAgICAgICAgICAgZmFjZV9hcmVhX3JhdGlvcy5hcHBlbmQoX2ZhY2VfYXJlYV9yYXRpbyhzZWxlY3RlZCwgZnJhbWUuc2hhcGUpKQoKICAgICAgICBpZiBsZW4oZW1iZWRkaW5ncykgPCBtaW5pbXVtX3ZhbGlkX2ZyYW1lczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIHsKICAgICAgICAgICAgICAgICJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwKICAgICAgICAgICAgICAgICJyZWFzb24iOiAiaW5zdWZmaWNpZW50X3ZhbGlkX2ZhY2VzIiwKICAgICAgICAgICAgICAgICJzYW1wbGVkX2ZyYW1lcyI6IGxlbihpbmRpY2VzKSwKICAgICAgICAgICAgICAgICJ2YWxpZF9mcmFtZXMiOiBsZW4oZW1iZWRkaW5ncyksCiAgICAgICAgICAgIH0KICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBWaWRlb0VtYmVkZGluZygKICAgICAgICAgICAgICAgIHN1YmplY3RfaWQ9cm93LnN1YmplY3RfaWQsCiAgICAgICAgICAgICAgICB2aWRlb19pZD1yb3cudmlkZW9faWQsCiAgICAgICAgICAgICAgICByZWxhdGl2ZV9wYXRoPXJvdy5yZWxhdGl2ZV9wYXRoLAogICAgICAgICAgICAgICAgZW1iZWRkaW5nPWwyX25vcm1hbGl6ZShucC5tZWFuKG5wLnN0YWNrKGVtYmVkZGluZ3MpLCBheGlzPTApKSwKICAgICAgICAgICAgICAgIHNhbXBsZWRfZnJhbWVzPWxlbihpbmRpY2VzKSwKICAgICAgICAgICAgICAgIHZhbGlkX2ZyYW1lcz1sZW4oZW1iZWRkaW5ncyksCiAgICAgICAgICAgICAgICBtZWFuX2RldGVjdGlvbl9zY29yZT1mbG9hdChucC5uYW5tZWFuKGRldGVjdGlvbl9zY29yZXMpKSwKICAgICAgICAgICAgICAgIG1lYW5fZmFjZV9hcmVhX3JhdGlvPWZsb2F0KG5wLm1lYW4oZmFjZV9hcmVhX3JhdGlvcykpLAogICAgICAgICAgICAgICAgZGVjb2RlX3NlY29uZHM9ZGVjb2RlX3NlY29uZHMsCiAgICAgICAgICAgICAgICBpbmZlcmVuY2Vfc2Vjb25kcz1pbmZlcmVuY2Vfc2Vjb25kcywKICAgICAgICAgICAgKSwKICAgICAgICAgICAgTm9uZSwKICAgICAgICApCiAgICBmaW5hbGx5OgogICAgICAgIGNhcHR1cmUucmVsZWFzZSgpCgoKZGVmIF9zaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgaGFuZGxlOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogaGFuZGxlLnJlYWQoMTAyNCAqIDEwMjQpLCBiIiIpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiBfZ2l0X2NvbW1pdCgpIC0+IHN0ciB8IE5vbmU6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KAogICAgICAgICAgICBbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgICAgIHN0ZGVycj1zdWJwcm9jZXNzLkRFVk5VTEwsCiAgICAgICAgKS5zdHJpcCgpCiAgICBleGNlcHQgKEZpbGVOb3RGb3VuZEVycm9yLCBzdWJwcm9jZXNzLlN1YnByb2Nlc3NFcnJvcik6CiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgX3dyaXRlX3JlamVjdHMocm93czogU2VxdWVuY2VbZGljdFtzdHIsIG9iamVjdF1dLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgaWYgbm90IHJvd3M6CiAgICAgICAgcmV0dXJuCiAgICBmaWVsZHMgPSBzb3J0ZWQoe2tleSBmb3Igcm93IGluIHJvd3MgZm9yIGtleSBpbiByb3d9KQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICIudG1wIikKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWZpZWxkcykKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwYXRoKQoKCmRlZiBfbW9kZWxfaGFzaGVzKG1vZGVsX3Jvb3Q6IFBhdGgsIG1vZGVsX25hbWU6IHN0cikgLT4gZGljdFtzdHIsIHN0cl06CiAgICBtb2RlbF9kaXIgPSBtb2RlbF9yb290LmV4cGFuZHVzZXIoKSAvICJtb2RlbHMiIC8gbW9kZWxfbmFtZQogICAgaWYgbm90IG1vZGVsX2Rpci5leGlzdHMoKToKICAgICAgICByZXR1cm4ge30KICAgIHJldHVybiB7CiAgICAgICAgc3RyKHBhdGgucmVsYXRpdmVfdG8obW9kZWxfZGlyKSk6IF9zaGEyNTYocGF0aCkKICAgICAgICBmb3IgcGF0aCBpbiBzb3J0ZWQobW9kZWxfZGlyLnJnbG9iKCIqLm9ubngiKSkKICAgIH0KCgpkZWYgaW5pdGlhbGl6ZV9mYWNlX2FwcChtb2RlbF9uYW1lOiBzdHIsIG1vZGVsX3Jvb3Q6IFBhdGgsIGRldF9zaXplOiBpbnQpIC0+IHR1cGxlW0FueSwgZGljdFtzdHIsIG9iamVjdF1dOgogICAgaW1wb3J0IGluc2lnaHRmYWNlICAjIHR5cGU6IGlnbm9yZQogICAgaW1wb3J0IG9ubnhydW50aW1lIGFzIG9ydCAgIyB0eXBlOiBpZ25vcmUKICAgIGZyb20gaW5zaWdodGZhY2UuYXBwIGltcG9ydCBGYWNlQW5hbHlzaXMgICMgdHlwZTogaWdub3JlCgogICAgYXZhaWxhYmxlID0gb3J0LmdldF9hdmFpbGFibGVfcHJvdmlkZXJzKCkKICAgIHByb3ZpZGVycyA9IFsKICAgICAgICBwcm92aWRlcgogICAgICAgIGZvciBwcm92aWRlciBpbiAoIkNVREFFeGVjdXRpb25Qcm92aWRlciIsICJDUFVFeGVjdXRpb25Qcm92aWRlciIpCiAgICAgICAgaWYgcHJvdmlkZXIgaW4gYXZhaWxhYmxlCiAgICBdCiAgICBpZiBub3QgcHJvdmlkZXJzOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5vIHN1cHBvcnRlZCBPTk5YIFJ1bnRpbWUgcHJvdmlkZXIgZm91bmQ6IHthdmFpbGFibGV9IikKICAgIGFwcCA9IEZhY2VBbmFseXNpcygKICAgICAgICBuYW1lPW1vZGVsX25hbWUsCiAgICAgICAgcm9vdD1zdHIobW9kZWxfcm9vdC5leHBhbmR1c2VyKCkpLAogICAgICAgIGFsbG93ZWRfbW9kdWxlcz1bImRldGVjdGlvbiIsICJyZWNvZ25pdGlvbiJdLAogICAgICAgIHByb3ZpZGVycz1wcm92aWRlcnMsCiAgICApCiAgICBjdWRhID0gIkNVREFFeGVjdXRpb25Qcm92aWRlciIgaW4gcHJvdmlkZXJzCiAgICBhcHAucHJlcGFyZSgKICAgICAgICBjdHhfaWQ9MCBpZiBjdWRhIGVsc2UgLTEsCiAgICAgICAgZGV0X3NpemU9KGRldF9zaXplLCBkZXRfc2l6ZSksCiAgICApCiAgICBpbnZlbnRvcnkgPSB7CiAgICAgICAgImluc2lnaHRmYWNlX3ZlcnNpb24iOiBnZXRhdHRyKGluc2lnaHRmYWNlLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpLAogICAgICAgICJvbm54cnVudGltZV92ZXJzaW9uIjogb3J0Ll9fdmVyc2lvbl9fLAogICAgICAgICJvbm54cnVudGltZV9hdmFpbGFibGVfcHJvdmlkZXJzIjogYXZhaWxhYmxlLAogICAgICAgICJvbm54cnVudGltZV9zZWxlY3RlZF9wcm92aWRlcnMiOiBwcm92aWRlcnMsCiAgICAgICAgImRldmljZSI6ICJjdWRhIiBpZiBjdWRhIGVsc2UgImNwdSIsCiAgICAgICAgIm1vZGVsX25hbWUiOiBtb2RlbF9uYW1lLAogICAgICAgICJtb2RlbF9yb290Ijogc3RyKG1vZGVsX3Jvb3QuZXhwYW5kdXNlcigpKSwKICAgICAgICAibW9kZWxfaGFzaGVzIjogX21vZGVsX2hhc2hlcyhtb2RlbF9yb290LCBtb2RlbF9uYW1lKSwKICAgIH0KICAgIHJldHVybiBhcHAsIGludmVudG9yeQoKCmRlZiBydW5fcGlwZWxpbmUoYXJnczogYXJncGFyc2UuTmFtZXNwYWNlKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGlmIG5vdCBhcmdzLmFjY2VwdF9ub25jb21tZXJjaWFsX21vZGVsX2xpY2Vuc2U6CiAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKAogICAgICAgICAgICAiSW5zaWdodEZhY2UtcHJvdmlkZWQgcHJldHJhaW5lZCBtb2RlbHMgYXJlIG5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHk7ICIKICAgICAgICAgICAgInBhc3MgLS1hY2NlcHQtbm9uY29tbWVyY2lhbC1tb2RlbC1saWNlbnNlIGFmdGVyIHJldmlld2luZyB0aGUgbGljZW5zZS4iCiAgICAgICAgKQogICAgbWFuaWZlc3Rfcm93cyA9IHJlYWRfbWFuaWZlc3QoYXJncy5tYW5pZmVzdCkKICAgIHNlbGVjdGVkX3Jvd3MgPSBtYW5pZmVzdF9yb3dzCiAgICBpZiBhcmdzLm1vZGUgPT0gInNtb2tlIjoKICAgICAgICBzZWxlY3RlZF9yb3dzID0gc2VsZWN0X3Ntb2tlX3Jvd3MoCiAgICAgICAgICAgIG1hbmlmZXN0X3Jvd3MsCiAgICAgICAgICAgIHN1YmplY3RzPWFyZ3Muc21va2Vfc3ViamVjdHMsCiAgICAgICAgICAgIHZpZGVvc19wZXJfc3ViamVjdD1hcmdzLnNtb2tlX3ZpZGVvc19wZXJfc3ViamVjdCwKICAgICAgICApCgogICAgZXhpc3Rpbmc6IGxpc3RbVmlkZW9FbWJlZGRpbmddID0gW10KICAgIGlmIGFyZ3Mub3V0cHV0LmV4aXN0cygpOgogICAgICAgIGV4aXN0aW5nID0gbG9hZF92aWRlb19lbWJlZGRpbmdzKGFyZ3Mub3V0cHV0KQogICAgY29tcGxldGVkID0ge3JlY29yZC52aWRlb19pZCBmb3IgcmVjb3JkIGluIGV4aXN0aW5nfQogICAgcmVjb3JkcyA9IGxpc3QoZXhpc3RpbmcpCiAgICByZWplY3RzOiBsaXN0W2RpY3Rbc3RyLCBvYmplY3RdXSA9IFtdCgogICAgZmFjZV9hcHAsIHJ1bnRpbWVfaW52ZW50b3J5ID0gaW5pdGlhbGl6ZV9mYWNlX2FwcCgKICAgICAgICBhcmdzLm1vZGVsX25hbWUsCiAgICAgICAgYXJncy5tb2RlbF9yb290LAogICAgICAgIGFyZ3MuZGV0X3NpemUsCiAgICApCiAgICBzdGFydGVkID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykKICAgIHByb2Nlc3NlZF9zaW5jZV9jaGVja3BvaW50ID0gMAogICAgYXR0ZW1wdGVkID0gMAogICAgZm9yIGluZGV4LCByb3cgaW4gZW51bWVyYXRlKHNlbGVjdGVkX3Jvd3MsIHN0YXJ0PTEpOgogICAgICAgIGlmIHJvdy52aWRlb19pZCBpbiBjb21wbGV0ZWQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYXR0ZW1wdGVkICs9IDEKICAgICAgICB2aWRlb19wYXRoID0gYXJncy52aWRlb19yb290IC8gUGF0aChyb3cucmVsYXRpdmVfcGF0aCkKICAgICAgICBpZiBub3QgdmlkZW9fcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmVqZWN0cy5hcHBlbmQoeyJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwgInJlYXNvbiI6ICJ2aWRlb19taXNzaW5nIn0pCiAgICAgICAgICAgIGlmIGFyZ3MuZmFpbF9mYXN0OgogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IodmlkZW9fcGF0aCkKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlY29yZCwgcmVqZWN0ID0gZW1iZWRfdmlkZW8oCiAgICAgICAgICAgICAgICB2aWRlb19wYXRoLAogICAgICAgICAgICAgICAgcm93LAogICAgICAgICAgICAgICAgZmFjZV9hcHAsCiAgICAgICAgICAgICAgICBmcmFtZXNfcGVyX3ZpZGVvPWFyZ3MuZnJhbWVzX3Blcl92aWRlbywKICAgICAgICAgICAgICAgIG1pbmltdW1fdmFsaWRfZnJhbWVzPWFyZ3MubWluaW11bV92YWxpZF9mcmFtZXMsCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgaWYgYXJncy5mYWlsX2Zhc3Q6CiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICByZWNvcmQgPSBOb25lCiAgICAgICAgICAgIHJlamVjdCA9IHsKICAgICAgICAgICAgICAgICJ2aWRlb19pZCI6IHJvdy52aWRlb19pZCwKICAgICAgICAgICAgICAgICJyZWFzb24iOiAidW5leHBlY3RlZF9lcnJvciIsCiAgICAgICAgICAgICAgICAiZXJyb3JfdHlwZSI6IHR5cGUoZXhjKS5fX25hbWVfXywKICAgICAgICAgICAgICAgICJtZXNzYWdlIjogc3RyKGV4YylbOjMwMF0sCiAgICAgICAgICAgIH0KICAgICAgICBpZiByZWNvcmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgY29tcGxldGVkLmFkZChyZWNvcmQudmlkZW9faWQpCiAgICAgICAgICAgIHByb2Nlc3NlZF9zaW5jZV9jaGVja3BvaW50ICs9IDEKICAgICAgICBpZiByZWplY3QgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJlamVjdHMuYXBwZW5kKHJlamVjdCkKCiAgICAgICAgaWYgcHJvY2Vzc2VkX3NpbmNlX2NoZWNrcG9pbnQgPj0gYXJncy5jaGVja3BvaW50X2V2ZXJ5OgogICAgICAgICAgICBzYXZlX3ZpZGVvX2VtYmVkZGluZ3MocmVjb3JkcywgYXJncy5vdXRwdXQpCiAgICAgICAgICAgIF93cml0ZV9yZWplY3RzKHJlamVjdHMsIGFyZ3MucmVqZWN0cykKICAgICAgICAgICAgcHJvY2Vzc2VkX3NpbmNlX2NoZWNrcG9pbnQgPSAwCiAgICAgICAgaWYgaW5kZXggPT0gMSBvciBpbmRleCAlIGFyZ3MucHJvZ3Jlc3NfZXZlcnkgPT0gMCBvciBpbmRleCA9PSBsZW4oc2VsZWN0ZWRfcm93cyk6CiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzZWxlY3RlZCI6IGxlbihzZWxlY3RlZF9yb3dzKSwKICAgICAgICAgICAgICAgICAgICAgICAgInZpc2l0ZWQiOiBpbmRleCwKICAgICAgICAgICAgICAgICAgICAgICAgInN1Y2Nlc3NmdWxfdG90YWwiOiBsZW4ocmVjb3JkcyksCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWplY3RlZF90aGlzX3J1biI6IGxlbihyZWplY3RzKSwKICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgICAgIGVuc3VyZV9hc2NpaT1GYWxzZSwKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICBmbHVzaD1UcnVlLAogICAgICAgICAgICApCgogICAgaWYgbm90IHJlY29yZHM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJubyB2aWRlbyBlbWJlZGRpbmdzIHdlcmUgcHJvZHVjZWQiKQogICAgc2F2ZV92aWRlb19lbWJlZGRpbmdzKHJlY29yZHMsIGFyZ3Mub3V0cHV0KQogICAgX3dyaXRlX3JlamVjdHMocmVqZWN0cywgYXJncy5yZWplY3RzKQogICAgZW5kZWQgPSBkYXRldGltZS5ub3codGltZXpvbmUudXRjKQogICAgcmVwb3J0ID0gewogICAgICAgICJzdGF0dXMiOiAiY29tcGxldGVkIiwKICAgICAgICAibW9kZSI6IGFyZ3MubW9kZSwKICAgICAgICAic2VsZWN0ZWRfdmlkZW9fY291bnQiOiBsZW4oc2VsZWN0ZWRfcm93cyksCiAgICAgICAgImF0dGVtcHRlZF90aGlzX3J1biI6IGF0dGVtcHRlZCwKICAgICAgICAic3VjY2Vzc2Z1bF92aWRlb19jb3VudF90b3RhbCI6IGxlbihyZWNvcmRzKSwKICAgICAgICAicmVqZWN0ZWRfdGhpc19ydW4iOiBsZW4ocmVqZWN0cyksCiAgICAgICAgImZyYW1lc19wZXJfdmlkZW8iOiBhcmdzLmZyYW1lc19wZXJfdmlkZW8sCiAgICAgICAgIm1pbmltdW1fdmFsaWRfZnJhbWVzIjogYXJncy5taW5pbXVtX3ZhbGlkX2ZyYW1lcywKICAgICAgICAic3RhcnRlZF91dGMiOiBzdGFydGVkLmlzb2Zvcm1hdCgpLAogICAgICAgICJlbmRlZF91dGMiOiBlbmRlZC5pc29mb3JtYXQoKSwKICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogKGVuZGVkIC0gc3RhcnRlZCkudG90YWxfc2Vjb25kcygpLAogICAgICAgICJtYW5pZmVzdCI6IHN0cihhcmdzLm1hbmlmZXN0KSwKICAgICAgICAibWFuaWZlc3Rfc2hhMjU2IjogX3NoYTI1NihhcmdzLm1hbmlmZXN0KSwKICAgICAgICAidmlkZW9fcm9vdCI6IHN0cihhcmdzLnZpZGVvX3Jvb3QpLAogICAgICAgICJvdXRwdXQiOiBzdHIoYXJncy5vdXRwdXQpLAogICAgICAgICJyZWplY3RzIjogc3RyKGFyZ3MucmVqZWN0cyksCiAgICAgICAgImdpdF9jb21taXQiOiBfZ2l0X2NvbW1pdCgpLAogICAgICAgICJtb2RlbF9saWNlbnNlX3Njb3BlIjogIkluc2lnaHRGYWNlLXByb3ZpZGVkIHdlaWdodHM6IG5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHkiLAogICAgICAgICoqcnVudGltZV9pbnZlbnRvcnksCiAgICB9CiAgICBhcmdzLnJ1bl9yZXBvcnQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGFyZ3MucnVuX3JlcG9ydC53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVwb3J0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQogICAgcmV0dXJuIHJlcG9ydAoKCmRlZiBidWlsZF9wYXJzZXIoKSAtPiBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1hbmlmZXN0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12aWRlby1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlamVjdHMiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1yZXBvcnQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGUiLCBjaG9pY2VzPSgic21va2UiLCAiZnVsbCIpLCBkZWZhdWx0PSJmdWxsIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc21va2Utc3ViamVjdHMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zbW9rZS12aWRlb3MtcGVyLXN1YmplY3QiLCB0eXBlPWludCwgZGVmYXVsdD0xKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mcmFtZXMtcGVyLXZpZGVvIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1pbmltdW0tdmFsaWQtZnJhbWVzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludC1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTI1KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm9ncmVzcy1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTEwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXQtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTY0MCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtbmFtZSIsIGRlZmF1bHQ9ImJ1ZmZhbG9fbCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsLXJvb3QiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgifi8uaW5zaWdodGZhY2UiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYWNjZXB0LW5vbmNvbW1lcmNpYWwtbW9kZWwtbGljZW5zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZhaWwtZmFzdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICByZXR1cm4gcGFyc2VyCgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBhcmdzID0gYnVpbGRfcGFyc2VyKCkucGFyc2VfYXJncygpCiAgICByZXBvcnQgPSBydW5fcGlwZWxpbmUoYXJncykKICAgIHByaW50KGpzb24uZHVtcHMocmVwb3J0LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo='}
EMBEDDED_CODE_SHA256 = "b8b679668101d878c076857d890be8ec8b091bda266034f490cd1ae89059c7a6"

if IN_HOSTED_COLAB and CODE_SOURCE == "github":
    REPO_DIR = Path("/content/face-image")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
elif IN_HOSTED_COLAB:
    REPO_DIR = Path("/content/face-image")
    for relative_path, encoded in EMBEDDED_FILES_B64.items():
        target = REPO_DIR / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(encoded))
    GIT_COMMIT = f"embedded:{EMBEDDED_CODE_SHA256[:12]}"
else:
    REPO_DIR = Path.cwd()
    try:
        GIT_COMMIT = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
        ).strip()
    except (FileNotFoundError, subprocess.SubprocessError):
        GIT_COMMIT = f"local:{EMBEDDED_CODE_SHA256[:12]}"

os.chdir(REPO_DIR)
print({"repo": str(REPO_DIR), "code_source": CODE_SOURCE, "code_version": GIT_COMMIT})

In [ ]:
#@title 4. Drive 연결, ZIP 확인, 작업 경로 설정
import shutil

if USE_GOOGLE_DRIVE:
    if not IN_HOSTED_COLAB:
        print("Local runtime: Google Drive mount cell is skipped.")
    else:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)

source_zip = Path(SOURCE_ZIP_PATH).expanduser()
if not source_zip.exists():
    raise FileNotFoundError(
        f"ZIP not found: {source_zip}. Upload the official Celeb-DF-v2.zip or correct SOURCE_ZIP_PATH."
    )

WORK_ROOT = Path("/content/celebdf_faceguard") if IN_HOSTED_COLAB else REPO_DIR / "outputs" / "celebdf_faceguard"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_ROOT = WORK_ROOT / "videos"
MANIFEST = WORK_ROOT / "celeb_real_manifest.csv"
INVENTORY_JSON = WORK_ROOT / "celeb_real_inventory.json"

if PERSIST_DERIVED_RESULTS_TO_DRIVE:
    if not I_CONFIRM_CELEBDF_CLOUD_PROCESSING_IS_ALLOWED:
        raise PermissionError("Derived embedding persistence also requires cloud-processing permission.")
    RESULT_ROOT = Path(DRIVE_RESULT_DIR)
else:
    RESULT_ROOT = WORK_ROOT / "results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

runtime_zip = source_zip
if COPY_ZIP_TO_RUNTIME:
    required = int(source_zip.stat().st_size * 1.25 + 3_000_000_000)
    free = shutil.disk_usage(WORK_ROOT).free
    if free < required:
        raise OSError(f"Not enough runtime disk: free={free}, required={required}")
    runtime_zip = WORK_ROOT / "Celeb-DF-v2.zip"
    if not runtime_zip.exists() or runtime_zip.stat().st_size != source_zip.stat().st_size:
        from tqdm.auto import tqdm
        temporary = runtime_zip.with_suffix(".zip.part")
        with source_zip.open("rb") as src, temporary.open("wb") as dst, tqdm(
            total=source_zip.stat().st_size, unit="B", unit_scale=True, desc="Copy ZIP"
        ) as progress:
            while chunk := src.read(8 * 1024 * 1024):
                dst.write(chunk)
                progress.update(len(chunk))
        temporary.replace(runtime_zip)

print({
    "source_zip": str(source_zip),
    "zip_gb": round(source_zip.stat().st_size / 1e9, 3),
    "runtime_zip": str(runtime_zip),
    "work_root": str(WORK_ROOT),
    "result_root": str(RESULT_ROOT),
    "runtime_free_gb": round(shutil.disk_usage(WORK_ROOT).free / 1e9, 2),
})

In [ ]:
#@title 5. ZIP 중앙 디렉터리 검사와 전체 590개 manifest 생성
import json

inventory_command = [
    sys.executable,
    "scripts/celebdf_faceguard.py",
    "inventory",
    str(runtime_zip),
    "--manifest", str(MANIFEST),
    "--summary", str(INVENTORY_JSON),
]
subprocess.run(inventory_command, check=True)
inventory = json.loads(INVENTORY_JSON.read_text(encoding="utf-8"))
assert inventory["video_count"] == 590, inventory
assert inventory["subject_count"] == 59, inventory
assert inventory["eligible_subjects_ge_8_videos"] == 56, inventory
print(json.dumps({
    key: inventory[key] for key in [
        "video_count", "subject_count", "uncompressed_bytes",
        "eligible_subjects_ge_8_videos", "excluded_subjects_lt_8_videos"
    ]
}, ensure_ascii=False, indent=2))

In [ ]:
#@title 6. Smoke 2개 영상 추출 후 Celeb-real 590개 전체 추출
if RUN_SMOKE_BEFORE_FULL:
    subprocess.run([
        sys.executable, "scripts/celebdf_faceguard.py", "extract", str(runtime_zip),
        "--manifest", str(MANIFEST), "--output", str(VIDEO_ROOT),
        "--mode", "smoke", "--smoke-subjects", "2", "--smoke-videos-per-subject", "1",
    ], check=True)

subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "extract", str(runtime_zip),
    "--manifest", str(MANIFEST), "--output", str(VIDEO_ROOT), "--mode", "full",
], check=True)
extracted_videos = sorted((VIDEO_ROOT / "Celeb-real").glob("*.mp4"))
if len(extracted_videos) != 590:
    raise RuntimeError(f"Expected 590 extracted videos, found {len(extracted_videos)}")
print({
    "extracted_videos": len(extracted_videos),
    "extracted_gb": round(sum(path.stat().st_size for path in extracted_videos) / 1e9, 3),
})

In [ ]:
#@title 7. GPU/ONNX Runtime 확인
import subprocess
import onnxruntime as ort

providers = ort.get_available_providers()
print({"onnxruntime": ort.__version__, "providers": providers})
if IN_HOSTED_COLAB and "CUDAExecutionProvider" not in providers:
    raise RuntimeError(
        "CUDAExecutionProvider is unavailable. Select a GPU runtime, then restart and rerun."
    )
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True))
except (FileNotFoundError, subprocess.SubprocessError):
    print("nvidia-smi unavailable; CPU/local runtime may be active.")

In [ ]:
#@title 8. Smoke 추론 — 설치·모델 다운로드·얼굴 탐지 확인
EMBEDDINGS_NPZ = RESULT_ROOT / "celeb_real_video_embeddings.npz"
REJECTS_CSV = RESULT_ROOT / "celeb_real_rejects.csv"
RUN_REPORT_JSON = RESULT_ROOT / "celeb_real_arcface_run.json"

base_runner_command = [
    sys.executable, "scripts/run_celebdf_arcface.py",
    "--manifest", str(MANIFEST),
    "--video-root", str(VIDEO_ROOT),
    "--output", str(EMBEDDINGS_NPZ),
    "--rejects", str(REJECTS_CSV),
    "--run-report", str(RUN_REPORT_JSON),
    "--frames-per-video", str(FRAMES_PER_VIDEO),
    "--minimum-valid-frames", str(MINIMUM_VALID_FRAMES),
    "--checkpoint-every", "25",
    "--model-name", "buffalo_l",
    "--accept-noncommercial-model-license",
]
if RUN_SMOKE_BEFORE_FULL:
    subprocess.run(
        base_runner_command + [
            "--mode", "smoke", "--smoke-subjects", "2", "--smoke-videos-per-subject", "1"
        ],
        check=True,
    )
print("Smoke inference completed. The full cell below resumes from the same NPZ checkpoint.")

In [ ]:
#@title 9. Celeb-real 590개 전체 ArcFace 실행
if RUN_FULL_590_VIDEOS:
    subprocess.run(base_runner_command + ["--mode", "full"], check=True)
else:
    raise RuntimeError("Full run was unexpectedly disabled.")
print(json.dumps(json.loads(RUN_REPORT_JSON.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

In [ ]:
#@title 10. 처리 품질 확인
import numpy as np
import pandas as pd

with np.load(EMBEDDINGS_NPZ, allow_pickle=False) as payload:
    quality = pd.DataFrame({
        "subject_id": payload["subject_ids"],
        "video_id": payload["video_ids"],
        "sampled_frames": payload["sampled_frames"],
        "valid_frames": payload["valid_frames"],
        "mean_detection_score": payload["mean_detection_scores"],
        "mean_face_area_ratio": payload["mean_face_area_ratios"],
        "decode_seconds": payload["decode_seconds"],
        "inference_seconds": payload["inference_seconds"],
    })
print({
    "successful_videos": len(quality),
    "success_rate": len(quality) / 590,
    "eligible_subjects_ge_8_successful_videos": int((quality.groupby("subject_id").size() >= 8).sum()),
})
display(quality.describe(include="all"))
display(quality.groupby("subject_id").size().sort_values().rename("successful_videos").to_frame())
if len(quality) < 560:
    print("WARNING: success rate is below the expected guardrail; inspect reject reasons before evaluation.")

In [ ]:
#@title 11. 등록 3장/5장 평가와 95% CI 생성
METRICS_JSON = RESULT_ROOT / "celeb_real_arcface_metrics.json"
subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "evaluate",
    "--embeddings", str(EMBEDDINGS_NPZ),
    "--output", str(METRICS_JSON),
    "--seed", str(SEED),
    "--bootstrap-repeats", str(BOOTSTRAP_REPEATS),
], check=True)
metrics = json.loads(METRICS_JSON.read_text(encoding="utf-8"))

metric_rows = []
for protocol_name, protocol in metrics["protocols"].items():
    row = {
        "protocol": protocol_name,
        "test_roc_auc": protocol["test_roc_auc"],
        "test_eer": protocol["test_eer"],
        "roc_auc_ci_low": protocol["roc_auc_95ci"][0],
        "roc_auc_ci_high": protocol["roc_auc_95ci"][1],
        "eer_ci_low": protocol["eer_95ci"][0],
        "eer_ci_high": protocol["eer_95ci"][1],
        "positive_pairs": protocol["test_positive_pairs"],
        "negative_pairs": protocol["test_negative_pairs"],
    }
    for far_key, point in protocol["operating_points"].items():
        row[f"{far_key}_threshold"] = point["threshold_selected_on_validation"]
        row[f"{far_key}_test_tar"] = point["test"]["tar"]
        row[f"{far_key}_test_far"] = point["test"]["far"]
        row[f"{far_key}_test_frr"] = point["test"]["frr"]
    metric_rows.append(row)
metrics_table = pd.DataFrame(metric_rows)
METRICS_CSV = RESULT_ROOT / "celeb_real_arcface_metrics.csv"
metrics_table.to_csv(METRICS_CSV, index=False)
display(metrics_table.T)
print({
    "eligible_subjects": metrics["eligible_subject_count"],
    "validation_subjects": metrics["validation_subject_count"],
    "test_subjects": metrics["test_subject_count"],
})

In [ ]:
#@title 12. ROC와 score 분포 그래프
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.insert(0, str(REPO_DIR / "scripts"))
from celebdf_faceguard import (
    auc_eer, build_pair_scores, group_eligible_records,
    load_video_embeddings, roc_curve, split_subjects,
)

records = load_video_embeddings(EMBEDDINGS_NPZ)
grouped = group_eligible_records(records, seed=SEED)
validation_subjects, test_subjects = split_subjects(grouped, seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for reference_count, color in [(3, "#2E74B5"), (5, "#E07A2D")]:
    pairs = build_pair_scores(grouped, test_subjects, reference_count=reference_count)
    fpr, tpr, _ = roc_curve(pairs.labels, pairs.scores)
    auc, eer = auc_eer(pairs.labels, pairs.scores)
    axes[0].semilogx(np.clip(fpr, 1e-5, 1), tpr, color=color, label=f"ref {reference_count} | AUC={auc:.4f}, EER={eer:.4f}")
    sample_negative = pairs.scores[pairs.labels == 0]
    if len(sample_negative) > 20000:
        rng = np.random.default_rng(SEED + reference_count)
        sample_negative = rng.choice(sample_negative, 20000, replace=False)
    sns.kdeplot(pairs.scores[pairs.labels == 1], ax=axes[1], color=color, linestyle="-", label=f"ref {reference_count} positive")
    sns.kdeplot(sample_negative, ax=axes[1], color=color, linestyle="--", label=f"ref {reference_count} negative")
axes[0].set(xlabel="False Accept Rate (log)", ylabel="True Accept Rate", title="Celeb-real identity verification ROC", xlim=(1e-5, 1), ylim=(0, 1.01))
axes[0].grid(True, alpha=0.25)
axes[0].legend()
axes[1].set(xlabel="Cosine similarity", ylabel="Density", title="Test score distributions")
axes[1].legend()
fig.tight_layout()
FIGURE_PNG = RESULT_ROOT / "celeb_real_arcface_roc_scores.png"
fig.savefig(FIGURE_PNG, dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
#@title 13. 비식별 결과 묶음 저장
import zipfile

RESULT_BUNDLE = RESULT_ROOT / "celeb_real_arcface_results.zip"
bundle_files = [
    INVENTORY_JSON, RUN_REPORT_JSON, METRICS_JSON, METRICS_CSV, FIGURE_PNG,
]
if REJECTS_CSV.exists():
    bundle_files.append(REJECTS_CSV)
with zipfile.ZipFile(RESULT_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        archive.write(path, arcname=path.name)
print({"result_bundle": str(RESULT_BUNDLE), "size_mb": round(RESULT_BUNDLE.stat().st_size / 1e6, 2)})

if IN_HOSTED_COLAB and not PERSIST_DERIVED_RESULTS_TO_DRIVE:
    from google.colab import files
    files.download(str(RESULT_BUNDLE))

## 결과 해석 시 주의사항

1. 이 결과는 사전학습 `buffalo_l` ArcFace baseline의 **Celeb-real 동일인 검증 성능**이다.
2. threshold는 validation 인물에서 고정한 뒤 겹치지 않는 test 인물에 적용한다. 운영 임계값은 실제 서비스 데이터로 다시 검증해야 한다.
3. Celeb-real은 한국인 전용 데이터가 아니다. AI-Hub 승인이 나면 같은 프로토콜을 한국인 안면 이미지에 재실행하여 일반화 차이를 비교한다.
4. 실제 얼굴 이미지나 프레임은 결과 묶음에 포함하지 않는다. `.npz` 임베딩은 생체정보로 취급하며 외부 공개·Git 커밋을 금지한다.
5. `buffalo_l` 제공 가중치는 비상업 연구 전용이다. 제품 배포 전 상업 사용 가능한 별도 모델·가중치를 선택한다.

공식 참고: [InsightFace PyPI](https://pypi.org/project/insightface/), [InsightFace GitHub](https://github.com/deepinsight/insightface)